# MNIST Classification using a Multi-Layer Perceptron (MLP)

In [1]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report


## Activation functions and loss

In [2]:
def relu(z):
    return np.maximum(0, z)


def relu_derivative(z):
    return (z > 0).astype(float)


def softmax(z):
    z_stable = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(z_stable)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)


def cross_entropy_loss(y_true, y_pred):
    eps = 1e-12
    y_pred = np.clip(y_pred, eps, 1.0 - eps)
    return -np.mean(np.sum(y_true * np.log(y_pred), axis=1))


## MLP class implemented from scratch

In [3]:
class MLP:
    def __init__(self, layer_sizes, learning_rate=0.01, seed=42):
        self.layer_sizes = layer_sizes
        self.learning_rate = learning_rate
        self.num_layers = len(layer_sizes) - 1
        self.params = {}
        np.random.seed(seed)
        self._initialize_parameters()

    def _initialize_parameters(self):
        for i in range(1, len(self.layer_sizes)):
            input_dim = self.layer_sizes[i - 1]
            output_dim = self.layer_sizes[i]
            self.params[f'W{i}'] = np.random.randn(input_dim, output_dim) * np.sqrt(2.0 / input_dim)
            self.params[f'b{i}'] = np.zeros((1, output_dim))

    def forward(self, X):
        activations = {'A0': X}
        zs = {}

        for i in range(1, self.num_layers + 1):
            W = self.params[f'W{i}']
            b = self.params[f'b{i}']
            Z = np.dot(activations[f'A{i-1}'], W) + b
            zs[f'Z{i}'] = Z

            if i == self.num_layers:
                A = softmax(Z)
            else:
                A = relu(Z)

            activations[f'A{i}'] = A

        return activations, zs

    def backward(self, X, y, activations, zs):
        m = X.shape[0]
        grads = {}

        dZ = activations[f'A{self.num_layers}'] - y

        for i in reversed(range(1, self.num_layers + 1)):
            A_prev = activations[f'A{i-1}']
            W = self.params[f'W{i}']

            grads[f'dW{i}'] = np.dot(A_prev.T, dZ) / m
            grads[f'db{i}'] = np.sum(dZ, axis=0, keepdims=True) / m

            if i > 1:
                dA_prev = np.dot(dZ, W.T)
                dZ = dA_prev * relu_derivative(zs[f'Z{i-1}'])

        return grads

    def update_parameters(self, grads):
        for i in range(1, self.num_layers + 1):
            self.params[f'W{i}'] -= self.learning_rate * grads[f'dW{i}']
            self.params[f'b{i}'] -= self.learning_rate * grads[f'db{i}']

    def fit(self, X_train, y_train, X_val=None, y_val=None, epochs=20, batch_size=64):
        n_samples = X_train.shape[0]

        for epoch in range(epochs):
            indices = np.random.permutation(n_samples)
            X_train_shuffled = X_train[indices]
            y_train_shuffled = y_train[indices]

            epoch_loss = 0.0

            for start in range(0, n_samples, batch_size):
                end = start + batch_size
                X_batch = X_train_shuffled[start:end]
                y_batch = y_train_shuffled[start:end]

                activations, zs = self.forward(X_batch)
                loss = cross_entropy_loss(y_batch, activations[f'A{self.num_layers}'])
                grads = self.backward(X_batch, y_batch, activations, zs)
                self.update_parameters(grads)

                epoch_loss += loss * X_batch.shape[0]

            epoch_loss /= n_samples

            train_preds = self.predict(X_train)
            train_labels = np.argmax(y_train, axis=1)
            train_acc = accuracy_score(train_labels, train_preds)

            if X_val is not None and y_val is not None:
                val_preds = self.predict(X_val)
                val_labels = np.argmax(y_val, axis=1)
                val_acc = accuracy_score(val_labels, val_preds)
                print(f'Epoch {epoch + 1:2d}/{epochs} | Loss: {epoch_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}')
            else:
                print(f'Epoch {epoch + 1:2d}/{epochs} | Loss: {epoch_loss:.4f} | Train Acc: {train_acc:.4f}')

    def predict_proba(self, X):
        activations, _ = self.forward(X)
        return activations[f'A{self.num_layers}']

    def predict(self, X):
        probs = self.predict_proba(X)
        return np.argmax(probs, axis=1)


## Load and preprocess the MNIST dataset

In [4]:
print('Loading MNIST dataset...')
mnist = fetch_openml('mnist_784', version=1, as_frame=False)

X = mnist.data.astype(np.float32)
y = mnist.target.astype(int)

# Normalizing pixel values to [0, 1]
X = X / 255.0

# Splitting data into train and test sets
X_train, X_test, y_train_raw, y_test_raw = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# One-hot encoding the output labels
try:
    encoder = OneHotEncoder(sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(sparse=False)

y_train = encoder.fit_transform(y_train_raw.reshape(-1, 1))
y_test = encoder.transform(y_test_raw.reshape(-1, 1))

print('Training data shape:', X_train.shape)
print('Test data shape:', X_test.shape)


Loading MNIST dataset...


C:\Users\bahad\anaconda3\Lib\site-packages\sklearn\datasets\_openml.py:1002: FutureWarning: The default value of `parser` will change from `'liac-arff'` to `'auto'` in 1.4. You can set `parser='auto'` to silence this warning. Therefore, an `ImportError` will be raised from 1.4 if the dataset is dense and pandas is not installed. Note that the pandas parser may return different data types. See the Notes Section in fetch_openml's API doc for details.
  warn(


Training data shape: (56000, 784)
Test data shape: (14000, 784)


## Train and compare different architectures

In [5]:
architectures = [
    [784, 64, 10],
    [784, 128, 10],
    [784, 128, 64, 10],
    [784, 256, 128, 10]
]

results = []

for arch in architectures:
    print('\n' + '=' * 60)
    print(f'Training MLP with architecture: {arch}')
    print('=' * 60)

    mlp = MLP(layer_sizes=arch, learning_rate=0.01, seed=42)
    mlp.fit(X_train, y_train, X_val=X_test, y_val=y_test, epochs=15, batch_size=128)

    y_pred = mlp.predict(X_test)
    test_acc = accuracy_score(y_test_raw, y_pred)
    results.append((arch, test_acc))

    print(f'\nTest Accuracy for architecture {arch}: {test_acc:.4f}')



Training MLP with architecture: [784, 64, 10]
Epoch  1/15 | Loss: 1.2339 | Train Acc: 0.8368 | Val Acc: 0.8406
Epoch  2/15 | Loss: 0.5720 | Train Acc: 0.8711 | Val Acc: 0.8729
Epoch  3/15 | Loss: 0.4537 | Train Acc: 0.8858 | Val Acc: 0.8844
Epoch  4/15 | Loss: 0.4015 | Train Acc: 0.8944 | Val Acc: 0.8933
Epoch  5/15 | Loss: 0.3705 | Train Acc: 0.9006 | Val Acc: 0.8978
Epoch  6/15 | Loss: 0.3490 | Train Acc: 0.9054 | Val Acc: 0.9024
Epoch  7/15 | Loss: 0.3327 | Train Acc: 0.9090 | Val Acc: 0.9044
Epoch  8/15 | Loss: 0.3198 | Train Acc: 0.9123 | Val Acc: 0.9074
Epoch  9/15 | Loss: 0.3089 | Train Acc: 0.9151 | Val Acc: 0.9092
Epoch 10/15 | Loss: 0.2995 | Train Acc: 0.9167 | Val Acc: 0.9111
Epoch 11/15 | Loss: 0.2912 | Train Acc: 0.9189 | Val Acc: 0.9141
Epoch 12/15 | Loss: 0.2837 | Train Acc: 0.9203 | Val Acc: 0.9150
Epoch 13/15 | Loss: 0.2769 | Train Acc: 0.9232 | Val Acc: 0.9179
Epoch 14/15 | Loss: 0.2703 | Train Acc: 0.9252 | Val Acc: 0.9194
Epoch 15/15 | Loss: 0.2644 | Train Acc: 0.9

## Display architecture comparison

In [6]:
print('\n' + '=' * 60)
print('Architecture Comparison')
print('=' * 60)

for arch, acc in results:
    print(f'Architecture {arch} -> Test Accuracy: {acc:.4f}')



Architecture Comparison
Architecture [784, 64, 10] -> Test Accuracy: 0.9207
Architecture [784, 128, 10] -> Test Accuracy: 0.9237
Architecture [784, 128, 64, 10] -> Test Accuracy: 0.9408
Architecture [784, 256, 128, 10] -> Test Accuracy: 0.9415


## Train the best architecture again and print full classification report

In [7]:
best_arch = max(results, key=lambda x: x[1])[0]
print('Best architecture:', best_arch)

best_mlp = MLP(layer_sizes=best_arch, learning_rate=0.01, seed=42)
best_mlp.fit(X_train, y_train, X_val=X_test, y_val=y_test, epochs=15, batch_size=128)

best_preds = best_mlp.predict(X_test)

print('\nFinal Classification Report:')
print(classification_report(y_test_raw, best_preds))


Best architecture: [784, 256, 128, 10]
Epoch  1/15 | Loss: 1.0889 | Train Acc: 0.8612 | Val Acc: 0.8588
Epoch  2/15 | Loss: 0.4559 | Train Acc: 0.8931 | Val Acc: 0.8890
Epoch  3/15 | Loss: 0.3632 | Train Acc: 0.9066 | Val Acc: 0.9016
Epoch  4/15 | Loss: 0.3206 | Train Acc: 0.9152 | Val Acc: 0.9101
Epoch  5/15 | Loss: 0.2932 | Train Acc: 0.9208 | Val Acc: 0.9144
Epoch  6/15 | Loss: 0.2730 | Train Acc: 0.9257 | Val Acc: 0.9204
Epoch  7/15 | Loss: 0.2567 | Train Acc: 0.9296 | Val Acc: 0.9250
Epoch  8/15 | Loss: 0.2428 | Train Acc: 0.9331 | Val Acc: 0.9274
Epoch  9/15 | Loss: 0.2309 | Train Acc: 0.9354 | Val Acc: 0.9283
Epoch 10/15 | Loss: 0.2207 | Train Acc: 0.9384 | Val Acc: 0.9328
Epoch 11/15 | Loss: 0.2113 | Train Acc: 0.9418 | Val Acc: 0.9349
Epoch 12/15 | Loss: 0.2031 | Train Acc: 0.9447 | Val Acc: 0.9372
Epoch 13/15 | Loss: 0.1951 | Train Acc: 0.9465 | Val Acc: 0.9385
Epoch 14/15 | Loss: 0.1878 | Train Acc: 0.9483 | Val Acc: 0.9399
Epoch 15/15 | Loss: 0.1814 | Train Acc: 0.9501 | Va